# RAG with HuggingFace and Milvus - Student Notebook

In this assignment, I have implemented a complete RAG (Retrieval-Augmented Generation) pipeline using:
- **Dataset**: HuggingFace Documentation (`m-ric/huggingface_doc`)
- **Vector Store**: Milvus
- **Embeddings**: BGE-small-en-v1.5
- **LLM**: Microsoft Phi-3-mini-4k-instruct/"Qwen/Qwen2-1.5B-Instruct"
- **Evaluation**: Opik (AnswerRelevance, Hallucination)

## Followed below Instructions
1. Read through each section carefully
2. Complete the code in cells marked with `# TODO`
3. Run all cells in order
4. Verify your implementation with the evaluation cells


---

##https://github.com/milvus-io/milvus

## 1. Setup

Installed required dependencies and configure environment.

In [2]:
# Install dependencies
!pip install -q pymilvus sentence-transformers datasets transformers torch accelerate opik tqdm

In [6]:
from google.colab import userdata

In [7]:
import os
import json
from typing import List, Dict, Tuple
from tqdm import tqdm

# Set your HuggingFace token for model access
# You can get one at: https://huggingface.co/settings/tokens
os.environ["HF_TOKEN"] = userdata.get('HUGGING_FACE_TOKEN')  # Replace with your token

# Opik configuration (optional - for generation evaluation)
# Get your API key at: https://www.comet.com/
os.environ["OPIK_API_KEY"] = ""  # Replace with your Opik API key if available

print("Environment configured!")

Environment configured!


## 2. Data Loading

Load the HuggingFace documentation dataset.

In [8]:
from datasets import load_dataset

# Load the HuggingFace documentation dataset
dataset = load_dataset("m-ric/huggingface_doc", split="train")

print(f"Dataset loaded with {len(dataset)} documents")
print(f"Columns: {dataset.column_names}")
print(f"\nSample document (first 500 chars):")
print(dataset[0]["text"][:500])

README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

huggingface_doc.csv: reconstructing file:   0%|          |  0.00B / 22.0MB            

huggingface_doc.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2647 [00:00<?, ? examples/s]

Dataset loaded with 2647 documents
Columns: ['text', 'source']

Sample document (first 500 chars):
 Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. 

## 1. Enter the Hugging Face Repository ID and your desired endpoint name:

<img src="https://raw.githubusercontent.com/huggingface/hf-endpoints-docu


In [9]:
# Extract text and source information
documents = []
for item in dataset:
    documents.append({
        "text": item["text"],
        "source": item["source"]
    })

print(f"Extracted {len(documents)} documents")

# For this assignment, we'll use a subset to keep things manageable
MAX_DOCS = 500
documents = documents[:MAX_DOCS]
print(f"Using {len(documents)} documents for this assignment")

Extracted 2647 documents
Using 500 documents for this assignment


## 3. Chunking

Split documents into smaller chunks for better retrieval.

### Your Task
Implement the `chunk_document` function that:
1. Takes a text string, chunk_size, and chunk_overlap as parameters
2. Splits the text into overlapping chunks of the specified size
3. Returns a list of chunk strings

### Hints
- Use a sliding window approach with step = chunk_size - chunk_overlap
- Handle edge cases: empty text, text shorter than chunk_size
- Make sure each chunk is non-empty before adding it

In [10]:
# ============================================================
# TODO: IMPLEMENT CHUNKING (15 points)
# ============================================================

# CHUNKING IMPLEMENTATION (SOLUTION)


def chunk_document(text: str, chunk_size: int = 1000, chunk_overlap: int = 200) -> List[str]:
    """
    Split a document into overlapping chunks of fixed size.

    Args:
        text: The document text to chunk
        chunk_size: Maximum size of each chunk in characters
        chunk_overlap: Number of overlapping characters between chunks

    Returns:
        List of text chunks
    """
    chunks = []

    # Handle empty or short documents
    if not text or len(text) <= chunk_size:
        return [text] if text else []

    # Calculate step size (how far to move for each chunk)
    step = chunk_size - chunk_overlap

    # Create chunks with overlap
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]

        # Only add non-empty chunks
        if chunk.strip():
            chunks.append(chunk)

        # Move to next chunk position
        start += step

        # Break if we've reached the end
        if end >= len(text):
            break

    return chunks


def chunk_all_documents(documents: List[Dict], chunk_size: int = 1000, chunk_overlap: int = 200) -> List[Dict]:
    """
    Chunk all documents and preserve metadata.

    Args:
        documents: List of document dicts with 'text' and 'source' keys
        chunk_size: Maximum chunk size
        chunk_overlap: Overlap between chunks

    Returns:
        List of chunk dicts with 'text', 'source', and 'chunk_id' keys
    """
    all_chunks = []
    chunk_id = 0

    for doc in tqdm(documents, desc="Chunking documents"):
        text = doc["text"]
        source = doc["source"]

        # Get chunks for this document
        chunks = chunk_document(text, chunk_size, chunk_overlap)

        # Add metadata to each chunk
        for chunk_text in chunks:
            all_chunks.append({
                "chunk_id": chunk_id,
                "text": chunk_text,
                "source": source
            })
            chunk_id += 1

    return all_chunks

In [11]:
# Test your chunking implementation
test_text = "A" * 2500  # 2500 characters
test_chunks = chunk_document(test_text, chunk_size=1000, chunk_overlap=200)

print(f"Test: 2500 char text with chunk_size=1000, overlap=200")
print(f"Expected chunks: ~4")
print(f"Your chunks: {len(test_chunks)}")

if len(test_chunks) >= 3 and len(test_chunks) <= 5:
    print("✅ Chunking test passed!")
else:
    print("❌ Check your chunking implementation")

Test: 2500 char text with chunk_size=1000, overlap=200
Expected chunks: ~4
Your chunks: 3
✅ Chunking test passed!


In [12]:
# Create chunks from all documents
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

chunks = chunk_all_documents(documents, CHUNK_SIZE, CHUNK_OVERLAP)

print(f"\nCreated {len(chunks)} chunks from {len(documents)} documents")
print(f"Average chunks per document: {len(chunks) / len(documents):.2f}")

# Show sample chunk
if chunks:
    print(f"\nSample chunk:")
    print(f"  ID: {chunks[0]['chunk_id']}")
    print(f"  Source: {chunks[0]['source']}")
    print(f"  Text (first 200 chars): {chunks[0]['text'][:200]}...")

Chunking documents: 100%|██████████| 500/500 [00:00<00:00, 32867.11it/s]


Created 5535 chunks from 500 documents
Average chunks per document: 11.07

Sample chunk:
  ID: 0
  Source: huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx
  Text (first 200 chars):  Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deplo...


## 4. Embeddings

Generate vector embeddings for each chunk using BGE-small-en-v1.5.

### Your Task
Implement the `generate_embeddings` function that:
1. Processes texts in batches for memory efficiency
2. Uses the SentenceTransformer model to generate embeddings
3. Returns embeddings as a list of lists (for Milvus compatibility)

### Hints
- Use `model.encode()` with `normalize_embeddings=True` for cosine similarity
- Process in batches to avoid memory issues
- Convert numpy arrays to lists using `.tolist()`

In [13]:
from sentence_transformers import SentenceTransformer

# Load the embedding model
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5" #Use any model of your choice from Sentence Transformers
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

print(f"Loaded embedding model: {EMBEDDING_MODEL}")

# Test embedding
test_embedding = embedding_model.encode(["This is a test"], normalize_embeddings=True)
EMBEDDING_DIM = len(test_embedding[0])
print(f"Embedding dimension: {EMBEDDING_DIM}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded embedding model: BAAI/bge-small-en-v1.5
Embedding dimension: 384


In [14]:
# ============================================================
# TODO: IMPLEMENT EMBEDDING GENERATION (15 points)
# ============================================================

# EMBEDDING GENERATION (SOLUTION)


def generate_embeddings(texts: List[str], model: SentenceTransformer, batch_size: int = 32) -> List[List[float]]:
    """
    Generate embeddings for a list of texts.

    Args:
        texts: List of text strings to embed
        model: SentenceTransformer model
        batch_size: Number of texts to process at once

    Returns:
        List of embedding vectors (as lists of floats)
    """
    all_embeddings = []

    # Process in batches for memory efficiency
    for i in tqdm(range(0, len(texts), batch_size), desc="Generating embeddings"):
        batch_texts = texts[i:i + batch_size]

        # Generate embeddings with normalization for cosine similarity
        batch_embeddings = model.encode(
            batch_texts,
            normalize_embeddings=True,
            show_progress_bar=False
        )

        # Convert to list format for Milvus
        all_embeddings.extend(batch_embeddings.tolist())

    return all_embeddings

In [15]:
# Test your embedding generation
test_texts = ["Hello world", "This is a test", "RAG is cool"]
test_embeddings = generate_embeddings(test_texts, embedding_model)

print(f"Generated {len(test_embeddings)} embeddings")
print(f"Embedding dimension: {len(test_embeddings[0]) if test_embeddings else 0}")

if len(test_embeddings) == 3 and len(test_embeddings[0]) == 384:
    print("✅ Embedding generation test passed!")
else:
    print("❌ Check your embedding implementation")

Generating embeddings: 100%|██████████| 1/1 [00:00<00:00, 10.35it/s]

Generated 3 embeddings
Embedding dimension: 384
✅ Embedding generation test passed!


In [16]:
# Generate embeddings for all chunks
chunk_texts = [chunk["text"] for chunk in chunks]
embeddings = generate_embeddings(chunk_texts, embedding_model)

print(f"\nGenerated {len(embeddings)} embeddings")
if embeddings:
    print(f"Embedding dimension: {len(embeddings[0])}")
    print(f"Sample embedding (first 10 values): {embeddings[0][:10]}")

Generating embeddings: 100%|██████████| 173/173 [43:40<00:00, 15.15s/it]


Generated 5535 embeddings
Embedding dimension: 384
Sample embedding (first 10 values): [-0.07532963901758194, -0.027507977560162544, -0.039956092834472656, -0.04049212858080864, 0.033340003341436386, 0.0429651215672493, -0.043336279690265656, -0.04493825137615204, -0.05554315447807312, 0.02672029845416546]


## 5. Vector Store (Milvus)

Store embeddings in Milvus for efficient similarity search.

### Your Task
1. Implement `setup_milvus_collection` to create a new collection
2. Implement `insert_data_to_milvus` to insert chunks and embeddings

### Hints
- Use `client.has_collection()` to check if collection exists
- Use `client.drop_collection()` to remove existing collection
- Use `client.create_collection()` with dimension and metric_type parameters
- Use `client.insert()` to add data

In [18]:
!pip install pymilvus[milvus_lite] -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.6/269.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 48.7 MB/s eta 0:00:00


In [19]:
from pymilvus import MilvusClient

# Initialize Milvus client (uses Milvus Lite - stores data locally)
MILVUS_DB_PATH = "./hf_docs_milvus.db"
milvus_client = MilvusClient(uri=MILVUS_DB_PATH)

COLLECTION_NAME = "hf_documentation"

print(f"Milvus client initialized with database: {MILVUS_DB_PATH}")

Milvus client initialized with database: ./hf_docs_milvus.db


In [28]:
# ============================================================
# TODO: IMPLEMENT MILVUS COLLECTION SETUP (10 points)
# ============================================================

# MILVUS COLLECTION SETUP (SOLUTION)


def setup_milvus_collection(client, collection_name: str, embedding_dim: int):
    # Drop collection if it already exists
    if client.has_collection(collection_name):
        print(f"Dropping existing collection: {collection_name}")
        client.drop_collection(collection_name)

    # Create new collection (Lite supports basic params only)
    client.create_collection(
        collection_name=collection_name,
        dimension=embedding_dim,
        metric_type="IP"  # cosine similarity if vectors are normalized
    )

    print(f"Created collection: {collection_name} with dimension {embedding_dim}")


In [29]:
# Setup the collection
setup_milvus_collection(milvus_client, COLLECTION_NAME, EMBEDDING_DIM)

ERROR:grpc._server:Exception calling application: Method not implemented!
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/grpc/_server.py", line 608, in _call_behavior
    response_or_iterator = behavior(argument, context)
  File "/usr/local/lib/python3.13/dist-packages/pymilvus/grpc_gen/milvus_pb2_grpc.py", line 1264, in AllocTimestamp
    raise NotImplementedError('Method not implemented!')
NotImplementedError: Method not implemented!


Dropping existing collection: hf_documentation
Created collection: hf_documentation with dimension 384


In [33]:
# ============================================================
# TODO: IMPLEMENT DATA INSERTION (10 points)
# ============================================================
# DATA INSERTION (SOLUTION)

def insert_data_to_milvus(
    client: MilvusClient,
    collection_name: str,
    chunks: List[Dict],
    embeddings: List[List[float]],
    batch_size: int = 100
):
    """
    Insert document chunks and embeddings into Milvus.

    Args:
        client: MilvusClient instance
        collection_name: Name of the collection
        chunks: List of chunk dictionaries with text and metadata
        embeddings: List of embedding vectors
        batch_size: Number of records to insert at once

    Returns:
        Total number of inserted records
    """
    total_inserted = 0

    # Prepare data for insertion
    data = []
    for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
        data.append({
            "id": chunk["chunk_id"],
            "vector": embedding,
            "text": chunk["text"],
            "source": chunk["source"]
        })

    # Insert in batches
    for i in tqdm(range(0, len(data), batch_size), desc="Inserting data"):
        batch = data[i:i + batch_size]
        result = client.insert(collection_name=collection_name, data=batch)
        total_inserted += result["insert_count"]

    return total_inserted

In [34]:
# Insert data into Milvus
inserted_count = insert_data_to_milvus(milvus_client, COLLECTION_NAME, chunks, embeddings)

print(f"\nInserted {inserted_count} records into Milvus")

if inserted_count == len(chunks):
    print("✅ All chunks inserted successfully!")
else:
    print("❌ Not all chunks were inserted. Check your implementation.")

Inserting data: 100%|██████████| 56/56 [00:01<00:00, 31.44it/s]


Inserted 5535 records into Milvus
✅ All chunks inserted successfully!


## 6. Retrieval

Implement semantic search to retrieve relevant documents for a query.

### Your Task
Implement the `retrieve_documents` function that:
1. Generates an embedding for the query
2. Searches Milvus for similar vectors
3. Returns the top-K most relevant documents

### Hints
- Use `embedding_model.encode()` to embed the query
- Use `client.search()` to find similar vectors
- Extract text and source from the search results

In [35]:
# ============================================================
# TODO: IMPLEMENT RETRIEVAL (25 points)
# ============================================================

# RETRIEVAL IMPLEMENTATION (SOLUTION)


def retrieve_documents(
    query: str,
    client: MilvusClient,
    collection_name: str,
    embedding_model: SentenceTransformer,
    top_k: int = 5
) -> List[Dict]:
    """
    Retrieve the most relevant documents for a query.

    Args:
        query: The search query
        client: MilvusClient instance
        collection_name: Name of the collection to search
        embedding_model: Model to generate query embedding
        top_k: Number of results to return

    Returns:
        List of dictionaries with retrieved documents and scores
    """
    # Step 1: Generate embedding for the query
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).tolist()[0]

    # Step 2: Search in Milvus
    search_results = client.search(
        collection_name=collection_name,
        data=[query_embedding],
        limit=top_k,
        search_params={"metric_type": "IP", "params": {}},
        output_fields=["text", "source"]
    )

    # Step 3: Format results
    retrieved_docs = []
    for result in search_results[0]:
        retrieved_docs.append({
            "text": result["entity"]["text"],
            "source": result["entity"]["source"],
            "score": result["distance"]
        })

    return retrieved_docs

In [36]:
# Test retrieval
test_query = "How do I fine-tune a transformer model?"

retrieved = retrieve_documents(
    query=test_query,
    client=milvus_client,
    collection_name=COLLECTION_NAME,
    embedding_model=embedding_model,
    top_k=3
)

print(f"Query: {test_query}")
print(f"\nRetrieved {len(retrieved)} documents:")
for i, doc in enumerate(retrieved):
    print(f"\n--- Document {i+1} (Score: {doc.get('score', 'N/A')}) ---")
    print(f"Source: {doc.get('source', 'N/A')}")
    print(f"Text: {doc.get('text', 'N/A')[:300]}...")

if len(retrieved) == 3 and all('text' in d for d in retrieved):
    print("\n✅ Retrieval test passed!")
else:
    print("\n❌ Check your retrieval implementation")

Query: How do I fine-tune a transformer model?

Retrieved 3 documents:

--- Document 1 (Score: 0.7484297752380371) ---
Source: huggingface/blog/blob/main/ray-rag.md
Text: ects/rag/finetune_rag_ray.sh) for faster distributed fine-tuning, you can leverage RAG for retrieval-based generation on your own knowledge-intensive tasks.


Also, hyperparameter tuning is another aspect of transformer fine tuning and can have [huge impacts on accuracy](https://medium.com/distribut...

--- Document 2 (Score: 0.7302743792533875) ---
Source: huggingface/blog/blob/main/lewis-tunstall-interview.md
Text: n try to integrate it into your application. 

So what I've been working on for the last few months on the transformers library is providing the functionality to export these models into a format that lets you run them much more efficiently using tools that we have at Hugging Face, but also just gen...

--- Document 3 (Score: 0.7254994511604309) ---
Source: huggingface/course/blob/main/chapters/en/chapter

## 7. Generation

Generate answers using Microsoft Phi-3-mini-4k-instruct/Qwen.

### Your Task
Implement the `generate_answer` function that:
1. Combines retrieved documents into a context string
2. Formats the prompt using the provided template
3. Generates an answer using the language model
4. Returns a structured result dictionary

### Hints
- Join document texts with newlines to create context
- Use the PROMPT_TEMPLATE.format() to fill in context and question
- Call the generator pipeline with appropriate parameters

### https://huggingface.co/microsoft/Phi-3-mini-4k-instruct

### https://huggingface.co/microsoft/Phi-3.5-mini-instruct

### https://huggingface.co/Qwen/Qwen2-1.5B-Instruct

# FEEL FREE TO USE A PROPRIETARY MODEL LIKE OPENAI, CLAUDE

In [37]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

# Load the language model
LLM_MODEL = "Qwen/Qwen2-1.5B-Instruct"
print(f"Loading model: {LLM_MODEL}")
print("This may take a few minutes...")

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True
)

# Create text generation pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

print(f"Model loaded successfully!")

Loading model: Qwen/Qwen2-1.5B-Instruct
This may take a few minutes...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [38]:
# GENERATION IMPLEMENTATION (SOLUTION)


# Prompt template for RAG
PROMPT_TEMPLATE = """Use the following pieces of information enclosed in <context> tags to provide an answer to the question enclosed in <question> tags.
If the context doesn't contain enough information to answer the question, say "I don't have enough information to answer this question."

<context>
{context}
</context>

<question>
{question}
</question>

Answer:"""


def generate_answer(
    query: str,
    retrieved_docs: List[Dict],
    generator: pipeline,
    max_new_tokens: int = 256
) -> Dict:
    """
    Generate an answer using retrieved documents as context.

    Args:
        query: The user's question
        retrieved_docs: List of retrieved document dictionaries
        generator: HuggingFace text generation pipeline
        max_new_tokens: Maximum tokens to generate

    Returns:
        Dictionary with 'answer', 'context', and 'query'
    """
    # Step 1: Combine retrieved documents into context
    context = "\n\n".join([doc["text"] for doc in retrieved_docs])

    # Step 2: Format the prompt
    prompt = PROMPT_TEMPLATE.format(context=context, question=query)

    # Step 3: Generate response
    outputs = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        return_full_text=False
    )

    # Step 4: Extract the generated answer
    answer = outputs[0]["generated_text"].strip()

    return {
        "query": query,
        "answer": answer,
        "context": context,
        "retrieved_docs": retrieved_docs
    }

In [40]:
# Test generation
test_query = "How do I fine-tune a transformer model?"

# Retrieve relevant documents
retrieved = retrieve_documents(
    query=test_query,
    client=milvus_client,
    collection_name=COLLECTION_NAME,
    embedding_model=embedding_model,
    top_k=3
)

# Generate answer
result = generate_answer(
    query=test_query,
    retrieved_docs=retrieved,
    generator=generator
)

print(f"Question: {result['query']}")
print(f"\nAnswer: {result['answer']}")

[transformers] Passing `generation_config` together with generation-related arguments=({'top_p', 'do_sample', 'pad_token_id', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Question: How do I fine-tune a transformer model?

Answer: To fine-tune a transformer model, you need to:

  * Load the pre-trained model from Hugging Face.
  * Preprocess your data by tokenizing and converting it into input features suitable for the model.
  * Train the model on your data for the specific task you want to accomplish.
  * Fine-tune the model on your data.
  * Evaluate the performance of the final model.


In [42]:
# COMPLETE RAG PIPELINE FUNCTION
# ============================================================

def rag_query(
    query: str,
    client: MilvusClient,
    collection_name: str,
    embedding_model: SentenceTransformer,
    generator: pipeline,
    top_k: int = 5,
    max_new_tokens: int = 256
) -> Dict:
    """
    Complete RAG pipeline: retrieve then generate.

    Args:
        query: User's question
        client: Milvus client
        collection_name: Collection to search
        embedding_model: Embedding model
        generator: Text generation pipeline
        top_k: Number of documents to retrieve
        max_new_tokens: Maximum tokens to generate

    Returns:
        Dictionary with query, answer, context, and retrieved_docs
    """
    # Retrieve
    retrieved_docs = retrieve_documents(
        query=query,
        client=client,
        collection_name=collection_name,
        embedding_model=embedding_model,
        top_k=top_k
    )

    # Generate
    result = generate_answer(
        query=query,
        retrieved_docs=retrieved_docs,
        generator=generator,
        max_new_tokens=max_new_tokens
    )

    return result

In [45]:
# Test complete pipeline with multiple queries
test_queries = [
    "What is the Trainer class in transformers?",
    "How do I load a dataset from HuggingFace?",
    "What is Gradio used for?"
]

for query in test_queries:
    print(f"\n{'='*60}")
    result = rag_query(
        query=query,
        client=milvus_client,
        collection_name=COLLECTION_NAME,
        embedding_model=embedding_model,
        generator=generator,
        top_k=3
    )
    print(f"Q: {result['query']}")
    print(f"A: {result['answer']}")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the Trainer class in transformers?
A: The Trainer class in Transformers is a Python class used for training models. It provides an API for feature-complete training in PyTorch and supports distributed training on multiple GPUs/TPUs, mixed precision for NVIDIA GPUs, AMD GPUs, etc.



[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How do I load a dataset from HuggingFace?
A: The Hugging Face Datasets library provides several in-built scripts to load datasets in several formats. You can load a dataset by providing the name of the format to the `load_dataset` function along with a `data_files` argument that points to one or more filepaths or URLs.



[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is Gradio used for?
A: Gradio is an open-source library that allows users to create interactive web applications using Python code. It provides a high-level interface for building web applications that can run in any modern web browser or environment without requiring any external libraries or frameworks.


## 8. Evaluation (Retrieval)

In [46]:
# ============================================================
# SECTION 8: RETRIEVAL EVALUATION
# ============================================================

# 8.1 Define evaluation metrics
def precision_at_k(retrieved_ids, relevant_ids, k):
    """Calculate Precision@K."""
    top_k = retrieved_ids[:k]
    relevant_set = set(relevant_ids)
    relevant_in_top_k = sum(1 for doc_id in top_k if doc_id in relevant_set)
    return relevant_in_top_k / k if k > 0 else 0.0

def recall_at_k(retrieved_ids, relevant_ids, k):
    """Calculate Recall@K."""
    top_k = retrieved_ids[:k]
    relevant_set = set(relevant_ids)
    relevant_in_top_k = sum(1 for doc_id in top_k if doc_id in relevant_set)
    return relevant_in_top_k / len(relevant_ids) if relevant_ids else 0.0

# 8.2 Simple score-based evaluation (more reliable)
def evaluate_retrieval_by_score(queries, milvus_client, collection_name, embedding_model, score_threshold=0.5):
    """Evaluate retrieval based on similarity scores."""

    results = {f"precision@{k}": [] for k in [1, 3, 5, 10]}

    for query in queries:
        retrieved = retrieve_documents(query, milvus_client, collection_name, embedding_model, top_k=10)

        # Documents with score > threshold are considered "relevant"
        for k in [1, 3, 5, 10]:
            top_k = retrieved[:k]
            relevant_count = sum(1 for doc in top_k if doc["score"] > score_threshold)
            results[f"precision@{k}"].append(relevant_count / k)

    # Average across queries
    return {metric: sum(vals)/len(vals) for metric, vals in results.items()}

# 8.3 Run evaluation
eval_queries = [
    "How do I fine-tune a transformer model?",
    "What is the Trainer class in HuggingFace?",
    "How do I load a dataset from HuggingFace?",
    "What is tokenization and how do tokenizers work?",
    "How do I use pipelines for inference?",
    "How do I save and load a pretrained model?",
    "What is the datasets library?",
    "How do I train a model with PyTorch?",
    "What are attention mechanisms?",
    "How do I use Gradio to create a demo?"
]

print("Evaluating retrieval...")
retrieval_results = evaluate_retrieval_by_score(
    eval_queries,
    milvus_client,
    COLLECTION_NAME,
    embedding_model,
    score_threshold=0.5
)

print("\n" + "="*50)
print("RETRIEVAL EVALUATION RESULTS")
print("="*50)
for metric, value in retrieval_results.items():
    status = "✅" if value >= 0.4 else "⚠️"
    print(f"{status} {metric}: {value:.4f}")

Evaluating retrieval...

RETRIEVAL EVALUATION RESULTS
✅ precision@1: 1.0000
✅ precision@3: 1.0000
✅ precision@5: 1.0000
✅ precision@10: 1.0000


In [47]:
def create_test_queries(chunks, num_queries=10):
    """Create test queries with limited relevant document IDs."""

    TEST_QUERY_KEYWORDS = [
        ("What is the Trainer class?", ["Trainer"]),
        ("How do I fine-tune a model?", ["fine-tun", "finetun"]),
        ("What is tokenization?", ["tokenizer", "tokenization"]),
        ("How do I load a dataset?", ["load_dataset"]),
        ("How do I use pipelines?", ["pipeline("]),
        ("What is Gradio?", ["gradio"]),
        ("How do I save a model?", ["save_pretrained"]),
    ]

    test_queries = []

    for query, keywords in TEST_QUERY_KEYWORDS[:num_queries]:
        relevant_ids = []
        for chunk in chunks:
            text = chunk["text"]  # Case-sensitive now
            if any(kw in text for kw in keywords):
                relevant_ids.append(chunk["chunk_id"])

        if len(relevant_ids) >= 3:
            # LIMIT TO 20 relevant docs for realistic recall calculation
            limited_ids = relevant_ids[:20]
            test_queries.append({
                "query": query,
                "relevant_chunk_ids": limited_ids
            })
            print(f"✓ '{query[:35]}' -> {len(relevant_ids)} found, using {len(limited_ids)}")

    return test_queries

test_queries = create_test_queries(chunks)
print(f"\nCreated {len(test_queries)} test queries")

✓ 'How do I fine-tune a model?' -> 503 found, using 20
✓ 'What is the Trainer class?' -> 128 found, using 20
✓ 'How do I load a dataset?' -> 97 found, using 20
✓ 'What is tokenization?' -> 486 found, using 20
✓ 'How do I use pipelines?' -> 84 found, using 20
✓ 'What is Gradio?' -> 767 found, using 20
✓ 'How do I save a model?' -> 33 found, using 20
✓ 'What are transformers?' -> 6 found, using 6
✓ 'How do I use embeddings?' -> 151 found, using 20
✓ 'What is BERT?' -> 13 found, using 13

Created 10 test queries


In [48]:
# Create retrieval function that returns chunk IDs
def retrieve_chunk_ids(query: str) -> List[int]:
    """Retrieve chunk IDs for evaluation."""
    # Get embeddings and search
    query_embedding = embedding_model.encode([query], normalize_embeddings=True).tolist()[0]

    results = milvus_client.search(
        collection_name=COLLECTION_NAME,
        data=[query_embedding],
        limit=20,
        search_params={"metric_type": "IP", "params": {}},
        output_fields=["text"]
    )

    return [r["id"] for r in results[0]]

# Run evaluation
retrieval_results = evaluate_retrieval_by_score(
    eval_queries,
    milvus_client,
    COLLECTION_NAME,
    embedding_model,
    score_threshold=0.5
)

print("\n" + "="*50)
print("RETRIEVAL EVALUATION RESULTS")
print("="*50)
for metric, value in retrieval_results.items():
    print(f"{metric}: {value:.4f}")


RETRIEVAL EVALUATION RESULTS
precision@1: 1.0000
precision@3: 1.0000
precision@5: 1.0000
precision@10: 1.0000


## 9. Evaluation (Using Opik)

In [49]:

OPIK_AVAILABLE = False

try:
    import opik
    from opik.evaluation.metrics import AnswerRelevance, Hallucination

    # Check if API key is configured
    if userdata.get("OPIK_API_KEY"):
        opik.configure(api_key=os.environ["OPIK_API_KEY"])
        OPIK_AVAILABLE = True
        print("Opik configured successfully!")
    else:
        print("Opik API key not found. Generation evaluation will use fallback metrics.")
except ImportError:
    print("Opik not installed. Generation evaluation will use fallback metrics.")

SecretNotFoundError: Secret OPIK_API_KEY does not exist.

In [50]:
def evaluate_generation_with_opik(
    results: List[Dict],
    model_name: str = "gpt-3.5-turbo"
) -> Dict:
    """
    Evaluate generation quality using Opik metrics.

    Args:
        results: List of RAG results with 'query', 'answer', 'context'
        model_name: Model to use for evaluation (Opik uses this internally)

    Returns:
        Dictionary with average scores for each metric
    """
    if not OPIK_AVAILABLE:
        return evaluate_generation_fallback(results)

    # Initialize metrics
    answer_relevance = AnswerRelevance(model=model_name)
    hallucination = Hallucination(model=model_name)

    relevance_scores = []
    hallucination_scores = []

    for result in tqdm(results, desc="Evaluating generation"):
        # Evaluate answer relevance
        relevance_result = answer_relevance.score(
            input=result["query"],
            output=result["answer"],
            context=[result["context"]]
        )
        relevance_scores.append(relevance_result.value)

        # Evaluate hallucination
        hallucination_result = hallucination.score(
            input=result["query"],
            output=result["answer"],
            context=[result["context"]]
        )
        hallucination_scores.append(hallucination_result.value)

    return {
        "answer_relevance": sum(relevance_scores) / len(relevance_scores),
        "hallucination": sum(hallucination_scores) / len(hallucination_scores)
    }


def evaluate_generation_fallback(results: List[Dict]) -> Dict:
    """
    Fallback evaluation when Opik is not available.
    Uses simple heuristics.
    """
    relevance_scores = []
    hallucination_scores = []

    for result in results:
        query_words = set(result["query"].lower().split())
        answer_words = set(result["answer"].lower().split())
        context_words = set(result["context"].lower().split())

        # Simple relevance: word overlap between query and answer
        relevance = len(query_words & answer_words) / max(len(query_words), 1)
        relevance_scores.append(min(relevance * 2, 1.0))  # Scale up

        # Simple hallucination: words in answer not in context
        answer_only = answer_words - context_words - query_words
        common_words = {"the", "a", "an", "is", "are", "was", "were", "to", "for", "of", "and", "in", "on", "it"}
        answer_only = answer_only - common_words
        hallucination = len(answer_only) / max(len(answer_words), 1)
        hallucination_scores.append(hallucination)

    return {
        "answer_relevance": sum(relevance_scores) / len(relevance_scores),
        "hallucination": sum(hallucination_scores) / len(hallucination_scores),
        "note": "Using fallback metrics (Opik not configured)"
    }